In [ ]:
import sys
sys.path.append("../")

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
import torch.optim as optim

from src.preprocessing.transforms import train_transform, eval_transform
from src.preprocessing.loaders import create_dataloaders

print("Environment ready")

In [ ]:
DATASET_DIR = "../plantwild_v2"

print("Dataset exists:", os.path.exists(DATASET_DIR))
print("Dataset path:", os.path.abspath(DATASET_DIR))

In [ ]:
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, "").count(os.sep)

    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

class_counts = Counter()

for root, dirs, files in os.walk(DATASET_DIR):
    for file in files:
        if os.path.splitext(file)[1].lower() in image_extensions:
            class_name = os.path.basename(root)
            class_counts[class_name] += 1

print("Total images:", sum(class_counts.values()))
print("Total classes:", len(class_counts))

In [ ]:
records = []

for root, dirs, files in os.walk(DATASET_DIR):
    for file in files:
        if os.path.splitext(file)[1].lower() in image_extensions:
            records.append({
                "filepath": os.path.join(root, file),
                "class": os.path.basename(root)
            })

df = pd.DataFrame(records)

print("Total images:", len(df))
print("Total classes:", df["class"].nunique())

In [ ]:
class_counts_df = (
    pd.DataFrame(
        class_counts.items(),
        columns=["class", "image_count"]
    )
    .sort_values("image_count")
)

print("Smallest classes:")
display(class_counts_df.head(15))

print("\nLargest classes:")
display(class_counts_df.tail(15))

In [ ]:
MIN_IMAGES = 15

class_counts = df["class"].value_counts()

valid_classes = class_counts[
    class_counts >= MIN_IMAGES
].index

df_clean = df[
    df["class"].isin(valid_classes)
].copy()

print("Clean images:", len(df_clean))
print("Clean classes:", df_clean["class"].nunique())

print("\nClasses still below threshold:")
print(df_clean["class"].value_counts().min())

In [ ]:
print("Minimum images/class:", class_counts_df["image_count"].min())
print("Maximum images/class:", class_counts_df["image_count"].max())

ratio = (
    class_counts_df["image_count"].max()
    / class_counts_df["image_count"].min()
)

print(f"Imbalance ratio: {ratio:.1f}x")

In [ ]:
missing_files = df_clean[
    ~df_clean["filepath"].apply(os.path.exists)
]

duplicate_paths = df_clean[
    df_clean["filepath"].duplicated()
]

print("Missing files:", len(missing_files))
print("Duplicate paths:", len(duplicate_paths))

In [ ]:
bad_files = []

for path in df_clean["filepath"]:
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception:
        bad_files.append(path)

print("Corrupted/unreadable images:", len(bad_files))

if bad_files:
    print("\nFirst few bad files:")
    print(bad_files[:10])

In [ ]:
print("FINAL DATA AUDIT ")
print("Total usable images:", len(df_clean))
print("Total usable classes:", df_clean["class"].nunique())
print("Missing files:", len(missing_files))
print("Duplicate paths:", len(duplicate_paths))
print("Corrupted images:", len(bad_files))

In [ ]:
os.makedirs("../data", exist_ok=True)

df_clean.to_csv(
    "../data/cleaned_metadata.csv",
    index=False
)

print("Saved: data/cleaned_metadata.csv")

In [ ]:
rng = np.random.RandomState(42)

train_parts = []
val_parts = []
test_parts = []

for class_name, group in df_clean.groupby("class"):
    group = group.sample(frac=1, random_state=42).reset_index(drop=True)

    n = len(group)

    n_test = max(1, round(n * 0.10))
    n_val = max(1, round(n * 0.10))

    # Make sure at least one image remains for training
    if n - n_val - n_test < 1:
        n_train = n - 2
        n_val = 1
        n_test = 1
    else:
        n_train = n - n_val - n_test

    train_parts.append(group.iloc[:n_train])
    val_parts.append(group.iloc[n_train:n_train + n_val])
    test_parts.append(group.iloc[n_train + n_val:])

train_df = pd.concat(train_parts).sample(frac=1, random_state=42).reset_index(drop=True)
val_df = pd.concat(val_parts).sample(frac=1, random_state=42).reset_index(drop=True)
test_df = pd.concat(test_parts).sample(frac=1, random_state=42).reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nClasses:")
print("Train:", train_df["class"].nunique())
print("Val:", val_df["class"].nunique())
print("Test:", test_df["class"].nunique())

In [ ]:
train_paths = set(train_df["filepath"])
val_paths = set(val_df["filepath"])
test_paths = set(test_df["filepath"])

print("Train ∩ Validation:", len(train_paths & val_paths))
print("Train ∩ Test:", len(train_paths & test_paths))
print("Validation ∩ Test:", len(val_paths & test_paths))

In [ ]:
train_df.to_csv("../data/train.csv", index=False)
val_df.to_csv("../data/val.csv", index=False)
test_df.to_csv("../data/test.csv", index=False)

print("Frozen splits saved successfully.")

In [ ]:
from src.dataset import PlantDataset

print("PlantDataset imported successfully!")

In [ ]:
print("Preprocessing pipeline loaded successfully!")
print("Image size:", 224)

In [ ]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
classes = sorted(train_df["class"].unique())

class_to_idx = {
    name: idx
    for idx, name in enumerate(classes)
}

idx_to_class = {
    idx: name
    for name, idx in class_to_idx.items()
}

print("Number of classes:", len(classes))
print("First 5 classes:", classes[:5])

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    train_df,
    val_df,
    test_df,
    class_to_idx,
    batch_size=32,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

print("Label range:", labels.min().item(), "to", labels.max().item())

In [ ]:
image = images[0].permute(1, 2, 0)

# Undo ImageNet normalization just for displaying it
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

image = image.numpy() * std + mean
image = np.clip(image, 0, 1)

plt.imshow(image)
plt.title(idx_to_class[labels[0].item()])
plt.axis("off")
plt.show()

In [ ]:
print("DATASET REPORT ")

print("Classes:", len(classes))
print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print("\nTrain class distribution:")
print(train_df["class"].value_counts().describe())

In [ ]:
# Combine the frozen splits to recover exact post-exclusion counts
all_clean_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# Exact per-class counts after exclusion
final_class_counts = (
    all_clean_df["class"]
    .value_counts()
    .sort_index()
)

# Classes excluded earlier
excluded_classes = pd.Series({
    "coffee black rot": 7,
    "peach rust": 8,
    "peach anthracnose": 13
})

print("===== EXCLUDED CLASSES =====")

for class_name, count in excluded_classes.items():
    print(f"{class_name}: {count} images")

print("\nReason: fewer than 15 images.")

print("\nFINAL DATASET")
print("Images:", len(all_clean_df))
print("Classes:", all_clean_df["class"].nunique())

print("\n FINAL PER-CLASS COUNTS ")
display(final_class_counts.to_frame("image_count"))

# Save reports
final_class_counts.to_csv(
    "../data/final_class_counts.csv",
    header=True
)

excluded_classes.to_csv(
    "../data/excluded_classes.csv",
    header=True
)

print("\nReports saved successfully.")

In [ ]:

train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")

classes = sorted(train_df["class"].unique())

class_to_idx = {
    name: idx for idx, name in enumerate(classes)
}

from src.preprocessing.loaders import create_dataloaders

train_loader, val_loader, test_loader = create_dataloaders(
    train_df,
    val_df,
    test_df,
    class_to_idx,
    batch_size=32,
    num_workers=0
)

from src.models.baseline_cnn import BaselineCNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BaselineCNN(num_classes=len(classes)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 5

start_time = time.time()

for epoch in range(EPOCHS):

    model.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        predictions = torch.argmax(outputs, dim=1)

        train_total += labels.size(0)
        train_correct += (predictions == labels).sum().item()

    train_accuracy = train_correct / train_total

    model.eval()
    val_predictions = []
    val_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            val_predictions.extend(predictions.cpu().numpy())
            val_labels.extend(labels.numpy())

    val_accuracy = sum(
        p == y for p, y in zip(val_predictions, val_labels)
    ) / len(val_labels)

    val_macro_f1 = f1_score(
        val_labels,
        val_predictions,
        average="macro"
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro-F1: {val_macro_f1:.4f}"
    )

training_time = time.time() - start_time

os.makedirs("../model", exist_ok=True)

torch.save(
    model.state_dict(),
    "../model/baseline_cnn.pth"
)

baseline_results = pd.DataFrame([{
    "model": "Baseline CNN",
    "epochs": EPOCHS,
    "parameters": sum(p.numel() for p in model.parameters()),
    "validation_accuracy": val_accuracy,
    "validation_macro_f1": val_macro_f1,
    "training_time_minutes": training_time / 60
}])

baseline_results.to_csv(
    "../data/baseline_results.csv",
    index=False
)

print("\n===== BASELINE COMPLETE =====")
print(baseline_results)
print("\nModel saved: ../model/baseline_cnn.pth")
print("Results saved: ../data/baseline_results.csv")

In [ ]:
from src.models.efficientnet import create_efficientnet

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

efficientnet = create_efficientnet(
    num_classes=len(classes)
).to(device)

print("Device:", device)
print("EfficientNet-B0 created successfully!")

In [ ]:
total_params = sum(
    p.numel() for p in efficientnet.parameters()
)

trainable_params = sum(
    p.numel()
    for p in efficientnet.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

In [ ]:
import torch.nn as nn


criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, efficientnet.parameters()),
    lr=0.001
)

EPOCHS = 5

start_time = time.time()

for epoch in range(EPOCHS):

    efficientnet.train()

    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = efficientnet(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        predictions = torch.argmax(outputs, dim=1)

        train_total += labels.size(0)
        train_correct += (predictions == labels).sum().item()

    train_accuracy = train_correct / train_total

    # Validation
    efficientnet.eval()

    val_predictions = []
    val_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)

            outputs = efficientnet(images)
            predictions = torch.argmax(outputs, dim=1)

            val_predictions.extend(predictions.cpu().numpy())
            val_labels.extend(labels.numpy())

    val_accuracy = sum(
        p == y for p, y in zip(val_predictions, val_labels)
    ) / len(val_labels)

    val_macro_f1 = f1_score(
        val_labels,
        val_predictions,
        average="macro"
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro-F1: {val_macro_f1:.4f}"
    )

training_time = time.time() - start_time

print(f"\nTraining time: {training_time/60:.2f} minutes")

In [ ]:
os.makedirs("../model", exist_ok=True)

torch.save(
    efficientnet.state_dict(),
    "../model/efficientnet_feature_extraction.pth"
)

print("EfficientNet feature-extraction model saved.")

In [ ]:
feature_results = pd.DataFrame([{
    "model": "EfficientNet-B0 Feature Extraction",
    "epochs": 5,
    "total_parameters": 4151020,
    "trainable_parameters": 143472,
    "validation_accuracy": 0.5287,
    "validation_macro_f1": 0.4599,
    "training_time_minutes": 34.55
}])

feature_results.to_csv(
    "../data/efficientnet_feature_results.csv",
    index=False
)

print("Results saved.")

In [ ]:
from src.models.efficientnet import create_efficientnet_finetune

finetune_model = create_efficientnet_finetune(
    num_classes=len(classes)
).to(device)

total_params = sum(p.numel() for p in finetune_model.parameters())

trainable_params = sum(
    p.numel()
    for p in finetune_model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

In [ ]:
import torch
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(classes)),
    y=train_df["class"].map(class_to_idx).values
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("Class weights created.")
print("Min weight:", class_weights.min().item())
print("Max weight:", class_weights.max().item())

In [ ]:

finetune_model = create_efficientnet_finetune(
    num_classes=len(classes)
).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, finetune_model.parameters()),
    lr=1e-4
)

EPOCHS = 5
best_macro_f1 = 0.0
history = []

start_time = time.time()

for epoch in range(EPOCHS):

    finetune_model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = finetune_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)

        predictions = torch.argmax(outputs, dim=1)

        train_total += labels.size(0)
        train_correct += (predictions == labels).sum().item()

    train_loss /= train_total
    train_accuracy = train_correct / train_total

    finetune_model.eval()

    val_loss = 0.0
    val_predictions = []
    val_labels = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = finetune_model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)

            predictions = torch.argmax(outputs, dim=1)

            val_predictions.extend(predictions.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_labels)

    val_accuracy = sum(
        p == y
        for p, y in zip(val_predictions, val_labels)
    ) / len(val_labels)

    val_macro_f1 = f1_score(
        val_labels,
        val_predictions,
        average="macro"
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_macro_f1
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro-F1: {val_macro_f1:.4f}"
    )

    if val_macro_f1 > best_macro_f1:

        best_macro_f1 = val_macro_f1

        torch.save(
            finetune_model.state_dict(),
            "../model/efficientnet_finetuned_best.pth"
        )

        print("Best model saved!")

training_time = time.time() - start_time

history_df = pd.DataFrame(history)

history_df.to_csv(
    "../data/efficientnet_finetune_history.csv",
    index=False
)

print("\n===== FINE-TUNING COMPLETE =====")
print(f"Best Validation Macro-F1: {best_macro_f1:.4f}")
print(f"Training time: {training_time/60:.2f} minutes")
print("Model saved: ../model/efficientnet_finetuned_best.pth")
print("History saved: ../data/efficientnet_finetune_history.csv")

In [ ]:
finetune_model.load_state_dict(
    torch.load(
        "../model/efficientnet_finetuned_best.pth",
        map_location=device
    )
)

finetune_model.eval()

test_predictions = []
test_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = finetune_model(images)
        predictions = torch.argmax(outputs, dim=1)

        test_predictions.extend(predictions.cpu().numpy())
        test_labels.extend(labels.numpy())

test_accuracy = accuracy_score(
    test_labels,
    test_predictions
)

test_precision = precision_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

test_macro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

print("===== TEST RESULTS =====")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"Macro Precision: {test_precision:.4f}")
print(f"Macro Recall: {test_recall:.4f}")
print(f"Macro F1: {test_macro_f1:.4f}")

In [ ]:
history_df = pd.read_csv("../data/efficientnet_finetune_history.csv")

print(history_df)

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    test_labels,
    test_predictions,
    target_names=classes,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

report_df.to_csv(
    "../data/test_classification_report.csv"
)

smallest_classes = (
    train_df["class"]
    .value_counts()
    .sort_values()
    .head(10)
    .index
)

smallest_report = report_df.loc[
    smallest_classes,
    ["precision", "recall", "f1-score", "support"]
]

print(smallest_report)

In [ ]:
counts = train_df["class"].value_counts()

print("Classes with <10 images:", (counts < 10).sum())
print("Classes with <20 images:", (counts < 20).sum())
print("Classes with <50 images:", (counts < 50).sum())
print("Smallest class:", counts.min())
print("Largest class:", counts.max())

In [ ]:
finetune_model = create_efficientnet_finetune(
    num_classes=len(classes)
).to(device)

finetune_model.load_state_dict(
    torch.load(
        "../model/efficientnet_finetuned_best.pth",
        map_location=device
    )
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, finetune_model.parameters()),
    lr=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.3,
    patience=2,
    min_lr=1e-6
)

EPOCHS = 10
PATIENCE = 5
best_macro_f1 = 0.4592
epochs_without_improvement = 0
history = []

start_time = time.time()

for epoch in range(EPOCHS):

    finetune_model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = finetune_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)

        predictions = torch.argmax(outputs, dim=1)

        train_total += labels.size(0)
        train_correct += (predictions == labels).sum().item()

    train_loss /= train_total
    train_accuracy = train_correct / train_total

    finetune_model.eval()

    val_loss = 0.0
    val_predictions = []
    val_labels = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = finetune_model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)

            predictions = torch.argmax(outputs, dim=1)

            val_predictions.extend(predictions.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_labels)

    val_accuracy = sum(
        p == y
        for p, y in zip(val_predictions, val_labels)
    ) / len(val_labels)

    val_macro_f1 = f1_score(
        val_labels,
        val_predictions,
        average="macro"
    )

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_macro_f1,
        "learning_rate": current_lr
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro-F1: {val_macro_f1:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    if val_macro_f1 > best_macro_f1:

        best_macro_f1 = val_macro_f1
        epochs_without_improvement = 0

        torch.save(
            finetune_model.state_dict(),
            "../model/efficientnet_finetuned_best.pth"
        )

        print("Best model saved!")

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:

        print("Early stopping triggered.")
        break

training_time = time.time() - start_time

history_df = pd.DataFrame(history)

history_df.to_csv(
    "../data/efficientnet_finetune_extended_history.csv",
    index=False
)

print("\n===== EXTENDED FINE-TUNING COMPLETE =====")
print(f"Best Validation Macro-F1: {best_macro_f1:.4f}")
print(f"Training time: {training_time/60:.2f} minutes")
print("Model saved: ../model/efficientnet_finetuned_best.pth")
print("History saved: ../data/efficientnet_finetune_extended_history.csv")

In [ ]:
finetune_model = create_efficientnet_finetune(
    num_classes=len(classes)
).to(device)

finetune_model.load_state_dict(
    torch.load(
        "../model/efficientnet_finetuned_best.pth",
        map_location=device
    )
)

finetune_model.eval()

test_predictions = []
test_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = finetune_model(images)
        predictions = torch.argmax(outputs, dim=1)

        test_predictions.extend(predictions.cpu().numpy())
        test_labels.extend(labels.numpy())

test_accuracy = accuracy_score(test_labels, test_predictions)
test_precision = precision_score(test_labels, test_predictions, average="macro", zero_division=0)
test_recall = recall_score(test_labels, test_predictions, average="macro", zero_division=0)
test_macro_f1 = f1_score(test_labels, test_predictions, average="macro", zero_division=0)

print("===== UPDATED TEST RESULTS =====")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"Macro Precision: {test_precision:.4f}")
print(f"Macro Recall: {test_recall:.4f}")
print(f"Macro F1: {test_macro_f1:.4f}")

In [ ]:
final_results = pd.DataFrame([{
    "model": "EfficientNet-B0 Fine-Tuned",
    "test_accuracy": test_accuracy,
    "test_macro_precision": test_precision,
    "test_macro_recall": test_recall,
    "test_macro_f1": test_macro_f1
}])

final_results.to_csv(
    "../data/final_test_results.csv",
    index=False
)

print(final_results)
print("\nSaved: ../data/final_test_results.csv")

In [ ]:

cm = confusion_matrix(
    test_labels,
    test_predictions
)

plt.figure(figsize=(5 , 5))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:


report = classification_report(
    test_labels,
    test_predictions,
    target_names=classes,
    output_dict=True,
    zero_division=0
)

per_class_df = pd.DataFrame(report).T

per_class_df = per_class_df.iloc[:-3]

per_class_df = per_class_df.sort_values(
    "f1-score",
    ascending=False
)

print(per_class_df)

In [ ]:
print("===== TOP 10 CLASSES =====")
print(per_class_df.head(10))

print("\n===== BOTTOM 10 CLASSES =====")
print(per_class_df.tail(10))

In [ ]:
print("===== LOW SUPPORT CLASSES =====")
print(
    per_class_df[per_class_df["support"] < 10]
    .sort_values("support")
)

In [ ]:
import numpy as np

errors = []

for true_idx in range(len(classes)):
    for pred_idx in range(len(classes)):
        if true_idx != pred_idx and cm[true_idx, pred_idx] > 0:
            errors.append({
                "true_class": classes[true_idx],
                "predicted_class": classes[pred_idx],
                "count": cm[true_idx, pred_idx]
            })

error_df = pd.DataFrame(errors).sort_values(
    "count",
    ascending=False
)

print("===== TOP CONFUSIONS =====")
print(error_df.head(20).to_string(index=False))

In [ ]:
finetune_model = create_efficientnet_finetune(
    num_classes=len(classes)
).to(device)

finetune_model.load_state_dict(
    torch.load(
        "../model/efficientnet_finetuned_best.pth",
        map_location=device
    )
)

finetune_model.eval()

target_layer = finetune_model.features[-1]

print("Grad-CAM model loaded.")
print("Target layer:", target_layer)

In [ ]:
import matplotlib.pyplot as plt
from src.models.gradcam import GradCAM
images, labels = next(iter(test_loader))

image = images[0].to(device)
true_label = labels[0].item()

output = finetune_model(image.unsqueeze(0))
predicted_label = torch.argmax(output, dim=1).item()

cam = GradCAM(
    finetune_model,
    target_layer
)

heatmap = cam.generate(
    image.unsqueeze(0),
    predicted_label
)

print("True:", classes[true_label])
print("Predicted:", classes[predicted_label])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 5))
plt.imshow(heatmap, cmap="jet")
plt.title(f"Grad-CAM: {classes[predicted_label]}")
plt.axis("off")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

img = images[0].permute(1, 2, 0).cpu().numpy()

img = (img - img.min()) / (img.max() - img.min())

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.imshow(heatmap, cmap="jet", alpha=0.45)
plt.title(
    f"True: {classes[true_label]}\n"
    f"Predicted: {classes[predicted_label]}"
)
plt.axis("off")
plt.show()

In [ ]:
finetune_model.eval()

wrong_image = None
wrong_true = None
wrong_pred = None

with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        outputs = finetune_model(batch_images)
        predictions = outputs.argmax(dim=1)

        wrong = predictions != batch_labels

        if wrong.any():
            idx = torch.where(wrong)[0][0]

            wrong_image = batch_images[idx].unsqueeze(0)
            wrong_true = batch_labels[idx].item()
            wrong_pred = predictions[idx].item()
            break

print("True:", classes[wrong_true])
print("Predicted:", classes[wrong_pred])

In [ ]:
cam = GradCAM(
    finetune_model,
    target_layer
)

wrong_heatmap = cam.generate(
    wrong_image,
    wrong_pred
)

wrong_img = wrong_image[0].permute(1, 2, 0).cpu().numpy()
wrong_img = (wrong_img - wrong_img.min()) / (
    wrong_img.max() - wrong_img.min()
)

plt.figure(figsize=(6, 6))
plt.imshow(wrong_img)
plt.imshow(wrong_heatmap, cmap="jet", alpha=0.45)
plt.title(
    f"True: {classes[wrong_true]}\n"
    f"Predicted: {classes[wrong_pred]}"
)
plt.axis("off")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

finetune_model.eval()

sample_images = []
sample_labels = []
sample_predictions = []

for batch_images, batch_labels in test_loader:
    batch_images = batch_images.to(device)
    batch_labels = batch_labels.to(device)

    with torch.no_grad():
        outputs = finetune_model(batch_images)
        predictions = outputs.argmax(dim=1)

    for i in range(len(batch_images)):
        sample_images.append(batch_images[i].unsqueeze(0))
        sample_labels.append(batch_labels[i].item())
        sample_predictions.append(predictions[i].item())

    if len(sample_images) >= 20:
        break

cam = GradCAM(finetune_model, target_layer)

fig, axes = plt.subplots(4, 5, figsize=(15, 12))

for i, ax in enumerate(axes.flat):
    image = sample_images[i]
    true_label = sample_labels[i]
    predicted_label = sample_predictions[i]

    heatmap = cam.generate(
        image,
        predicted_label
    )

    img = image[0].permute(1, 2, 0).cpu().numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    ax.imshow(img)
    ax.imshow(heatmap, cmap="jet", alpha=0.4)
    ax.set_title(
        f"T: {classes[true_label]}\nP: {classes[predicted_label]}",
        fontsize=7
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image
import pytesseract
import pandas as pd

df = test_loader.dataset.dataframe

audit = []

for _, row in df.iterrows():
    path = row["filepath"]
    label = row["class"]

    try:
        image = Image.open(path).convert("RGB")
        text = pytesseract.image_to_string(image).strip()

        audit.append({
            "filepath": path,
            "class": label,
            "detected_text": text,
            "has_text": len(text) > 0
        })

    except Exception:
        audit.append({
            "filepath": path,
            "class": label,
            "detected_text": "",
            "has_text": False
        })

audit_df = pd.DataFrame(audit)

print("Total images:", len(audit_df))
print("Images with detected text:", audit_df["has_text"].sum())
print(f"Text percentage: {audit_df['has_text'].mean() * 100:.2f}%")

In [ ]:
comparison = pd.DataFrame([
    {
        "Model": "Baseline CNN",
        "Validation Accuracy": 0.1054,
        "Validation Macro-F1": 0.0358,
        "Test Accuracy": None,
        "Test Macro-F1": None
    },
    {
        "Model": "EfficientNet-B0 Feature Extraction",
        "Validation Accuracy": 0.5287,
        "Validation Macro-F1": 0.4599,
        "Test Accuracy": 0.5192,
        "Test Macro-F1": 0.4510
    },
    {
        "Model": "EfficientNet-B0 Fine-Tuned + Class Weights",
        "Validation Accuracy": None,
        "Validation Macro-F1": 0.5320,
        "Test Accuracy": 0.6054,
        "Test Macro-F1": 0.5543
    }
])

print(comparison.to_string(index=False))

In [ ]:
comparison.to_csv(
    "../data/model_comparison.csv",
    index=False
)

print("Saved: ../data/model_comparison.csv")

In [ ]:
result = predictor.predict(sample_path)

print("Image:", sample_path)

for i, prediction in enumerate(result, start=1):
    print(
        f"{i}. {prediction['class']} — "
        f"{prediction['confidence'] * 100:.2f}%"
    )

In [ ]:
with open("../data/classes.txt", "w", encoding="utf-8") as f:
    for class_name in classes:
        f.write(class_name + "\n")

print("Saved:", len(classes), "classes")